# 01. Group Aggregate EDA

EMS group 단위 전기·열·기상 연결성을 확인한다.

1차 범위는 `central_cooling`이다. 00 노트북에서 확인한 DB 입력 신뢰성 결과를 기준으로 1h mart를 사용한다.

## 1. 분석 기준

### 목적

중앙 냉각 계통의 전력 사용량, 냉각 열량, 기상 외생 변수의 관계를 확인한다.

### 대상

| 구분 | meter | measurement |
|---|---|---|
| 전기 | `H1.Z16`, `H1.Z11`, `H1.Z12`, `H1.Z24`, `H1.Z25` | `P` |
| 열 | `V.K21` | `P`, `Tdiff`, `Trl`, `Tvl`, `qv`, `W` |
| 기상 | `WeatherStation.Weather` | `Ta`, `Igc`, `Ah` |

### 출력 정책

기본 실행은 노트북 출력 중심으로 수행한다. `SAVE_OUTPUTS = True` 설정 시 검토용 산출물을 저장한다.

### Measurement 해석

`V.K21`은 중앙 냉각 열 계통 meter이다. `V.K21`의 `P`는 measurement dictionary 기준 `Active power`, 단위 `W`로 등록되어 있다. 이 노트북에서는 `cooling_thermal__P`로 표기한다. 열 계통의 순간 부하 규모를 나타내는 값으로 해석한다.

In [ ]:
# C01. 환경 설정 및 경로
from pathlib import Path
from datetime import datetime, timezone
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    display = print

sns.set_theme(style="whitegrid")
import koreanize_matplotlib
plt.rcParams["axes.unicode_minus"] = False

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    for candidate in [ROOT, *ROOT.parents]:
        if (candidate / "pyproject.toml").exists() and candidate.name == "EMS":
            ROOT = candidate
            break
os.chdir(ROOT)

SAVE_OUTPUTS = False
OUT_TABLE = ROOT / "outputs" / "tables" / "group_aggregate_eda"
OUT_FIG = ROOT / "outputs" / "figures" / "group_aggregate_eda"
if SAVE_OUTPUTS:
    OUT_TABLE.mkdir(parents=True, exist_ok=True)
    OUT_FIG.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}")
print(f"SAVE_OUTPUTS={SAVE_OUTPUTS}")
print(f"executed_at_utc={datetime.now(timezone.utc).isoformat()}")

In [ ]:
# C02. DB helper
import psycopg


def load_dotenv(path: Path) -> None:
    if path.exists():
        for raw_line in path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if line and "=" in line and line[:1] != "#":
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_dotenv(ROOT / ".env")
REQUIRED_ENV = ["DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD"]
missing_env = [key for key in REQUIRED_ENV if key not in os.environ]
if missing_env:
    raise RuntimeError(f"DB 환경변수 누락: {missing_env}")


def connect():
    return psycopg.connect(
        host=os.environ["DB_HOST"],
        port=os.environ["DB_PORT"],
        dbname=os.environ["DB_NAME"],
        user=os.environ["DB_USER"],
        password=os.environ["DB_PASSWORD"],
        connect_timeout=8,
    )


def query_df(sql: str, params=None, timeout_s: int = 60) -> pd.DataFrame:
    with connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SET statement_timeout = '{int(timeout_s)}s'")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            return pd.read_sql(sql, conn, params=params)


def show_table(df: pd.DataFrame, name: str) -> None:
    rows = len(df)
    if SAVE_OUTPUTS:
        OUT_TABLE.mkdir(parents=True, exist_ok=True)
        out = OUT_TABLE / name
        df.to_csv(out, index=False)
        print(f"saved={out} rows={rows}")
    else:
        print(f"table={name} rows={rows}")
    if rows == 0:
        display(pd.DataFrame({"table": [name], "rows": [0], "status": ["empty"]}))
    else:
        display(df)

In [ ]:
# C03. Group 정의
CENTRAL_COOLING_ELECTRIC = ["H1.Z16", "H1.Z11", "H1.Z12", "H1.Z24", "H1.Z25"]
COOLING_THERMAL = ["V.K21"]
WEATHER = ["WeatherStation.Weather"]

analysis_scope = pd.DataFrame([
    {"scope": "central_cooling_electric", "meter_urn": meter, "measurement": "P", "role": "electricity_consumption"}
    for meter in CENTRAL_COOLING_ELECTRIC
] + [
    {"scope": "cooling_thermal", "meter_urn": "V.K21", "measurement": measurement, "role": "thermal_flow"}
    for measurement in ["P", "Tdiff", "Trl", "Tvl", "qv", "W"]
] + [
    {"scope": "weather", "meter_urn": "WeatherStation.Weather", "measurement": measurement, "role": "external"}
    for measurement in ["Ta", "Igc", "Ah"]
])
show_table(analysis_scope, "analysis_scope.csv")

In [ ]:
# C04. Measurement 사전 확인
measurement_dict = query_df("""
SELECT measurement, unit, measurement_family, description
FROM ems.full_measurement_definition
WHERE measurement = ANY(%s)
ORDER BY measurement
""", params=(sorted(analysis_scope["measurement"].unique().tolist()),))
show_table(measurement_dict, "measurement_dictionary_subset.csv")

In [ ]:
# C05. Coverage 확인
coverage = query_df("""
SELECT
    meter_urn,
    measurement,
    resolution_code,
    status,
    count(*) AS source_files,
    sum(csv_rows) AS csv_rows,
    sum(inserted_rows) AS inserted_rows,
    sum(null_value_rows) AS null_value_rows
FROM ems.full_source_file
WHERE processing_level = 'corrected_resampled'
  AND resolution_code = '1h'
  AND meter_urn = ANY(%s)
  AND measurement = ANY(%s)
GROUP BY meter_urn, measurement, resolution_code, status
ORDER BY meter_urn, measurement
""", params=(analysis_scope["meter_urn"].unique().tolist(), analysis_scope["measurement"].unique().tolist()))
show_table(coverage, "coverage_1h.csv")

In [ ]:
# C06. 1h 데이터 로드
raw = query_df("""
SELECT ts, meter_urn, measurement, value
FROM ems.cr_measurement_1h
WHERE meter_urn = ANY(%s)
  AND measurement = ANY(%s)
ORDER BY ts, meter_urn, measurement
""", params=(analysis_scope["meter_urn"].unique().tolist(), analysis_scope["measurement"].unique().tolist()), timeout_s=120)
raw["ts"] = pd.to_datetime(raw["ts"], utc=True)
show_table(raw.head(20), "raw_preview.csv")
print(f"raw_rows={len(raw):,}")
print(f"ts_min={raw['ts'].min()}")
print(f"ts_max={raw['ts'].max()}")

In [ ]:
# C07. Wide table 구성
pivot = raw.pivot_table(index="ts", columns=["meter_urn", "measurement"], values="value", aggfunc="mean").sort_index()
wide = pd.DataFrame(index=pivot.index)

p_cols = [(meter, "P") for meter in CENTRAL_COOLING_ELECTRIC if (meter, "P") in pivot.columns]
wide["central_cooling__electric_P_sum"] = pivot[p_cols].sum(axis=1, min_count=1)
wide["central_cooling__electric_meter_count"] = pivot[p_cols].count(axis=1)

for measurement in ["P", "Tdiff", "Trl", "Tvl", "qv", "W"]:
    col = ("V.K21", measurement)
    if col in pivot.columns:
        wide[f"cooling_thermal__{measurement}"] = pivot[col]

for measurement in ["Ta", "Igc", "Ah"]:
    col = ("WeatherStation.Weather", measurement)
    if col in pivot.columns:
        wide[f"weather__{measurement}"] = pivot[col]

wide = wide.reset_index()
wide["year"] = wide["ts"].dt.year
wide["month"] = wide["ts"].dt.month
wide["hour"] = wide["ts"].dt.hour
wide["day_of_week"] = wide["ts"].dt.dayofweek
show_table(wide.head(20), "wide_preview.csv")
print(f"wide_rows={len(wide):,}")
print(f"wide_columns={len(wide.columns)}")

In [ ]:
# C08. 결측률 요약
missing_summary = (
    wide.drop(columns=["ts", "year", "month", "hour", "day_of_week"])
    .isna()
    .mean()
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "series"})
    .sort_values("missing_rate", ascending=False)
)
show_table(missing_summary, "missing_summary.csv")

In [ ]:
# C09. 월별 profile
monthly_profile = (
    wide.groupby(["year", "month"], dropna=False)
    .agg(
        electric_P_mean=("central_cooling__electric_P_sum", "mean"),
        electric_P_median=("central_cooling__electric_P_sum", "median"),
        thermal_P_mean=("cooling_thermal__P", "mean"),
        thermal_Tdiff_mean=("cooling_thermal__Tdiff", "mean"),
        weather_Ta_mean=("weather__Ta", "mean"),
        weather_Igc_mean=("weather__Igc", "mean"),
        rows=("ts", "count"),
    )
    .reset_index()
)
show_table(monthly_profile.head(24), "monthly_profile_preview.csv")

In [ ]:
# C10. 시간대 profile
hourly_profile = (
    wide.groupby(["month", "hour"], dropna=False)
    .agg(
        electric_P_mean=("central_cooling__electric_P_sum", "mean"),
        thermal_P_mean=("cooling_thermal__P", "mean"),
        weather_Ta_mean=("weather__Ta", "mean"),
        weather_Igc_mean=("weather__Igc", "mean"),
        rows=("ts", "count"),
    )
    .reset_index()
)
show_table(hourly_profile.head(24), "hourly_profile_preview.csv")

In [ ]:
# C11. 상관 요약
corr_cols = [
    "central_cooling__electric_P_sum",
    "cooling_thermal__P",
    "cooling_thermal__Tdiff",
    "cooling_thermal__qv",
    "weather__Ta",
    "weather__Igc",
    "weather__Ah",
]
corr_cols = [col for col in corr_cols if col in wide.columns]
corr = wide[corr_cols].corr(method="pearson").round(3)
show_table(corr.reset_index().rename(columns={"index": "series"}), "correlation_summary.csv")

In [ ]:
# C12. 기본 시각화
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

monthly_plot = monthly_profile.copy()
monthly_plot["date"] = pd.to_datetime(monthly_plot["year"].astype(str) + "-" + monthly_plot["month"].astype(str) + "-01")
axes[0, 0].plot(monthly_plot["date"], monthly_plot["electric_P_mean"], label="electric P mean")
axes[0, 0].set_title("Central cooling electric power monthly mean")
axes[0, 0].tick_params(axis="x", rotation=45)

axes[0, 1].plot(monthly_plot["date"], monthly_plot["thermal_P_mean"], label="thermal P mean", color="tab:orange")
axes[0, 1].set_title("Cooling thermal P monthly mean")
axes[0, 1].tick_params(axis="x", rotation=45)

sns.scatterplot(data=wide.sample(min(len(wide), 20000), random_state=42), x="weather__Ta", y="central_cooling__electric_P_sum", s=8, alpha=0.25, ax=axes[1, 0])
axes[1, 0].set_title("Electric P vs weather Ta")

sns.scatterplot(data=wide.sample(min(len(wide), 20000), random_state=43), x="cooling_thermal__P", y="central_cooling__electric_P_sum", s=8, alpha=0.25, ax=axes[1, 1])
axes[1, 1].set_title("Electric P vs thermal P")

fig.tight_layout()
if SAVE_OUTPUTS:
    OUT_FIG.mkdir(parents=True, exist_ok=True)
    fig_path = OUT_FIG / "central_cooling_overview.png"
    fig.savefig(fig_path, dpi=150)
    print(f"saved={fig_path}")
plt.show()

In [ ]:
# C13. Cooling season 구분
season_map = {
    1: "winter",
    2: "winter",
    3: "spring",
    4: "shoulder",
    5: "shoulder",
    6: "cooling_peak",
    7: "cooling_peak",
    8: "cooling_peak",
    9: "cooling_peak",
    10: "shoulder",
    11: "winter",
    12: "winter",
}
wide["cooling_season"] = wide["month"].map(season_map)
season_profile = (
    wide.groupby("cooling_season", dropna=False)
    .agg(
        electric_P_mean=("central_cooling__electric_P_sum", "mean"),
        electric_P_median=("central_cooling__electric_P_sum", "median"),
        thermal_P_mean=("cooling_thermal__P", "mean"),
        weather_Ta_mean=("weather__Ta", "mean"),
        weather_Igc_mean=("weather__Igc", "mean"),
        rows=("ts", "count"),
    )
    .reset_index()
    .sort_values("electric_P_mean", ascending=False)
)
show_table(season_profile, "season_profile.csv")

In [ ]:
# C14. 월별 시간대 heatmap
heatmap_data = wide.pivot_table(
    index="month",
    columns="hour",
    values="central_cooling__electric_P_sum",
    aggfunc="mean",
)
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heatmap_data, cmap="YlOrRd", ax=ax)
ax.set_title("Central cooling electric power by month and hour")
ax.set_xlabel("hour")
ax.set_ylabel("month")
fig.tight_layout()
if SAVE_OUTPUTS:
    OUT_FIG.mkdir(parents=True, exist_ok=True)
    fig_path = OUT_FIG / "central_cooling_month_hour_heatmap.png"
    fig.savefig(fig_path, dpi=150)
    print(f"saved={fig_path}")
plt.show()

In [ ]:
# C15. Ta bin별 전력 분포
bins = [-np.inf, 0, 5, 10, 15, 20, 25, np.inf]
labels = ["<=0", "0-5", "5-10", "10-15", "15-20", "20-25", ">25"]
wide["Ta_bin"] = pd.cut(wide["weather__Ta"], bins=bins, labels=labels)
ta_bin_profile = (
    wide.groupby("Ta_bin", observed=True)
    .agg(
        electric_P_mean=("central_cooling__electric_P_sum", "mean"),
        electric_P_median=("central_cooling__electric_P_sum", "median"),
        electric_P_q90=("central_cooling__electric_P_sum", lambda x: x.quantile(0.90)),
        thermal_P_mean=("cooling_thermal__P", "mean"),
        rows=("ts", "count"),
    )
    .reset_index()
)
show_table(ta_bin_profile, "ta_bin_profile.csv")

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=wide, x="Ta_bin", y="central_cooling__electric_P_sum", showfliers=False, ax=ax)
ax.set_title("Central cooling electric power by Ta bin")
ax.set_xlabel("Ta bin")
ax.set_ylabel("electric P sum")
fig.tight_layout()
if SAVE_OUTPUTS:
    OUT_FIG.mkdir(parents=True, exist_ok=True)
    fig_path = OUT_FIG / "central_cooling_ta_bin_boxplot.png"
    fig.savefig(fig_path, dpi=150)
    print(f"saved={fig_path}")
plt.show()

In [ ]:
# C16. 전력-열량 ratio
ratio_df = wide[["ts", "month", "hour", "central_cooling__electric_P_sum", "cooling_thermal__P", "weather__Ta"]].copy()
ratio_df["electric_to_thermal_ratio"] = np.where(
    ratio_df["cooling_thermal__P"].abs() > 1e-9,
    ratio_df["central_cooling__electric_P_sum"] / ratio_df["cooling_thermal__P"],
    np.nan,
)
ratio_summary = (
    ratio_df.groupby("month", dropna=False)
    .agg(
        ratio_median=("electric_to_thermal_ratio", "median"),
        ratio_q10=("electric_to_thermal_ratio", lambda x: x.quantile(0.10)),
        ratio_q90=("electric_to_thermal_ratio", lambda x: x.quantile(0.90)),
        electric_P_mean=("central_cooling__electric_P_sum", "mean"),
        thermal_P_mean=("cooling_thermal__P", "mean"),
        rows=("ts", "count"),
    )
    .reset_index()
)
show_table(ratio_summary, "electric_to_thermal_ratio_by_month.csv")

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=ratio_summary, x="month", y="ratio_median", marker="o", ax=ax)
ax.fill_between(ratio_summary["month"], ratio_summary["ratio_q10"], ratio_summary["ratio_q90"], alpha=0.2)
ax.set_title("Electric to thermal ratio by month")
ax.set_xlabel("month")
ax.set_ylabel("ratio")
fig.tight_layout()
if SAVE_OUTPUTS:
    OUT_FIG.mkdir(parents=True, exist_ok=True)
    fig_path = OUT_FIG / "central_cooling_ratio_by_month.png"
    fig.savefig(fig_path, dpi=150)
    print(f"saved={fig_path}")
plt.show()

In [ ]:
# C17. High-load 구간 확인
threshold = wide["central_cooling__electric_P_sum"].quantile(0.95)
high_load = wide[wide["central_cooling__electric_P_sum"] >= threshold].copy()
high_load_summary = (
    high_load.groupby(["month", "hour"], dropna=False)
    .agg(
        rows=("ts", "count"),
        electric_P_mean=("central_cooling__electric_P_sum", "mean"),
        thermal_P_mean=("cooling_thermal__P", "mean"),
        weather_Ta_mean=("weather__Ta", "mean"),
        weather_Igc_mean=("weather__Igc", "mean"),
    )
    .reset_index()
    .sort_values(["rows", "electric_P_mean"], ascending=[False, False])
)
print(f"high_load_threshold_q95={threshold:,.3f}")
show_table(high_load_summary.head(30), "high_load_month_hour_top30.csv")

fig, ax = plt.subplots(figsize=(12, 5))
sns.countplot(data=high_load, x="month", ax=ax, color="tab:red")
ax.set_title("High-load hours by month")
ax.set_xlabel("month")
ax.set_ylabel("hours")
fig.tight_layout()
if SAVE_OUTPUTS:
    OUT_FIG.mkdir(parents=True, exist_ok=True)
    fig_path = OUT_FIG / "central_cooling_high_load_by_month.png"
    fig.savefig(fig_path, dpi=150)
    print(f"saved={fig_path}")
plt.show()

In [ ]:
# C18. Cooling degree 관계 점검
base_temperatures = [15, 18, 20]
for base in base_temperatures:
    wide[f"cooling_degree_base_{base}"] = (wide["weather__Ta"] - base).clip(lower=0)

cooling_degree_profile = (
    wide.groupby("month", dropna=False)
    .agg(
        electric_P_mean=("central_cooling__electric_P_sum", "mean"),
        Ta_mean=("weather__Ta", "mean"),
        cooling_degree_15_mean=("cooling_degree_base_15", "mean"),
        cooling_degree_18_mean=("cooling_degree_base_18", "mean"),
        cooling_degree_20_mean=("cooling_degree_base_20", "mean"),
        rows=("ts", "count"),
    )
    .reset_index()
)
show_table(cooling_degree_profile, "cooling_degree_profile.csv")

cooling_degree_corr = wide[[
    "central_cooling__electric_P_sum",
    "cooling_degree_base_15",
    "cooling_degree_base_18",
    "cooling_degree_base_20",
]].corr().round(3)
show_table(cooling_degree_corr.reset_index().rename(columns={"index": "series"}), "cooling_degree_correlation.csv")

In [ ]:
# C19. 시간차 관계 점검
lag_series = [
    "weather__Ta",
    "weather__Igc",
    "weather__Ah",
    "cooling_thermal__P",
]
lag_hours = [0, 1, 2, 3, 6, 12, 24]
lag_rows = []
for series_name in lag_series:
    for lag in lag_hours:
        lagged = wide[series_name].shift(lag)
        corr_value = wide["central_cooling__electric_P_sum"].corr(lagged)
        lag_rows.append({"series": series_name, "lag_hours": lag, "corr_with_electric_P": corr_value})
lag_corr = pd.DataFrame(lag_rows).sort_values(["series", "lag_hours"])
show_table(lag_corr, "lag_correlation.csv")

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=lag_corr, x="lag_hours", y="corr_with_electric_P", hue="series", marker="o", ax=ax)
ax.set_title("시간차별 central cooling 전력 연결성")
ax.set_xlabel("lag hours")
ax.set_ylabel("correlation")
fig.tight_layout()
if SAVE_OUTPUTS:
    OUT_FIG.mkdir(parents=True, exist_ok=True)
    fig_path = OUT_FIG / "central_cooling_lag_correlation.png"
    fig.savefig(fig_path, dpi=150)
    print(f"saved={fig_path}")
plt.show()

In [ ]:
# C20. 구간별 연결성 점검
rolling_window = 24 * 30
rolling_min_periods = 24 * 7
rolling_df = wide[["ts", "central_cooling__electric_P_sum", "cooling_thermal__P", "weather__Ta", "weather__Igc"]].copy()
rolling_df["corr_electric_thermal_30d"] = rolling_df["central_cooling__electric_P_sum"].rolling(rolling_window, min_periods=rolling_min_periods).corr(rolling_df["cooling_thermal__P"])
rolling_df["corr_electric_Ta_30d"] = rolling_df["central_cooling__electric_P_sum"].rolling(rolling_window, min_periods=rolling_min_periods).corr(rolling_df["weather__Ta"])
rolling_df["corr_electric_Igc_30d"] = rolling_df["central_cooling__electric_P_sum"].rolling(rolling_window, min_periods=rolling_min_periods).corr(rolling_df["weather__Igc"])
rolling_corr_summary = rolling_df[["corr_electric_thermal_30d", "corr_electric_Ta_30d", "corr_electric_Igc_30d"]].describe(percentiles=[0.1, 0.5, 0.9]).round(3).reset_index()
show_table(rolling_corr_summary, "rolling_corr_summary.csv")

fig, ax = plt.subplots(figsize=(14, 5))
plot_df = rolling_df.set_index("ts")[["corr_electric_thermal_30d", "corr_electric_Ta_30d", "corr_electric_Igc_30d"]]
plot_df.plot(ax=ax, alpha=0.8)
ax.set_title("30일 이동 상관")
ax.set_xlabel("ts")
ax.set_ylabel("correlation")
fig.tight_layout()
if SAVE_OUTPUTS:
    OUT_FIG.mkdir(parents=True, exist_ok=True)
    fig_path = OUT_FIG / "central_cooling_rolling_correlation.png"
    fig.savefig(fig_path, dpi=150)
    print(f"saved={fig_path}")
plt.show()

In [ ]:
# C21. Weekday-hour profile
weekday_labels = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}
wide["weekday_label"] = wide["day_of_week"].map(weekday_labels)
weekday_hour_profile = (
    wide.groupby(["day_of_week", "weekday_label", "hour"], dropna=False)
    .agg(
        electric_P_mean=("central_cooling__electric_P_sum", "mean"),
        thermal_P_mean=("cooling_thermal__P", "mean"),
        weather_Ta_mean=("weather__Ta", "mean"),
        rows=("ts", "count"),
    )
    .reset_index()
)
show_table(weekday_hour_profile.head(30), "weekday_hour_profile_preview.csv")

weekday_heatmap = weekday_hour_profile.pivot_table(index="weekday_label", columns="hour", values="electric_P_mean", aggfunc="mean")
weekday_heatmap = weekday_heatmap.reindex(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(weekday_heatmap, cmap="YlOrRd", ax=ax)
ax.set_title("Central cooling electric power by weekday and hour")
ax.set_xlabel("hour")
ax.set_ylabel("weekday")
fig.tight_layout()
if SAVE_OUTPUTS:
    OUT_FIG.mkdir(parents=True, exist_ok=True)
    fig_path = OUT_FIG / "central_cooling_weekday_hour_heatmap.png"
    fig.savefig(fig_path, dpi=150)
    print(f"saved={fig_path}")
plt.show()

In [ ]:
# C22. High-load episode 요약
episode_df = wide[["ts", "month", "hour", "central_cooling__electric_P_sum", "cooling_thermal__P", "weather__Ta", "weather__Igc"]].copy()
episode_df["is_high_load"] = episode_df["central_cooling__electric_P_sum"] >= threshold
episode_df["episode_id"] = (episode_df["is_high_load"] != episode_df["is_high_load"].shift()).cumsum()
high_episode_summary = (
    episode_df[episode_df["is_high_load"]]
    .groupby("episode_id", dropna=False)
    .agg(
        start_ts=("ts", "min"),
        end_ts=("ts", "max"),
        duration_hours=("ts", "count"),
        max_electric_P=("central_cooling__electric_P_sum", "max"),
        mean_electric_P=("central_cooling__electric_P_sum", "mean"),
        mean_thermal_P=("cooling_thermal__P", "mean"),
        mean_Ta=("weather__Ta", "mean"),
        mean_Igc=("weather__Igc", "mean"),
    )
    .reset_index()
    .sort_values(["duration_hours", "max_electric_P"], ascending=[False, False])
)
show_table(high_episode_summary.head(30), "high_load_episode_top30.csv")

episode_duration_summary = high_episode_summary["duration_hours"].describe(percentiles=[0.5, 0.75, 0.9, 0.95]).round(2).reset_index()
show_table(episode_duration_summary, "high_load_episode_duration_summary.csv")

## 2. 검토 기준

1. `coverage_1h`에서 대상 meter와 measurement의 row 수를 확인한다.
2. `missing_summary`에서 집계 신호 결측률을 확인한다.
3. `monthly_profile`에서 계절 패턴을 확인한다.
4. `hourly_profile`에서 운영 시간대 패턴을 확인한다.
5. `correlation_summary`와 scatter plot으로 전기·열·기상 연결성을 확인한다.
6. `season_profile`에서 cooling season별 부하 수준을 확인한다.
7. `ta_bin_profile`에서 기온 구간별 전력 분포를 확인한다.
8. `electric_to_thermal_ratio_by_month`에서 전력-열량 비율의 월별 안정성을 확인한다.
9. `high_load_month_hour_top30`에서 고부하 시간대의 월·시간 집중도를 확인한다.
10. `cooling_degree_correlation`에서 cooling degree와 전력의 관계를 확인한다.
11. `lag_correlation`에서 외생 변수와 열 부하의 시간 지연 관계를 확인한다.
12. `rolling_corr_summary`에서 연결성의 시간 안정성을 확인한다.
13. `weekday_hour_profile`에서 요일·시간대 운영 패턴을 확인한다.
14. `high_load_episode_top30`에서 고부하 episode 지속 시간을 확인한다.
15. central_cooling 형식이 안정적이면 server, CHP, PV group으로 확장한다.